In [1]:
import yaml
import pandas as pd
import os
import sys
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix)

current_dir = os.path.dirname(os.path.abspath('.'))
project_root = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.insert(0, project_root)

In [2]:
from functions.feature_selection import SelectSingleFeature

In [3]:
# 1. Carregar configurações
with open(os.path.join("../config/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

# pipeline selection    
with open(os.path.join("../config/pipeline.yaml"), "r") as f:
    config_pipe = yaml.safe_load(f)

# model selection    
with open(os.path.join( "../config/model.yaml"), "r") as f:
    config_model = yaml.safe_load(f)

In [4]:
# Get feature eng data
pipeline_name = "Pipeline3"
model_name = "rf"

X_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"X_val_feat_eng_{pipeline_name}.parquet")
)

y_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"y_val_feat_eng_{pipeline_name}.parquet")
)



In [5]:
X_val.drop(
    columns = config_model['single_model']['cols_2_drop'],
    inplace=True
) 

In [6]:
model_path = os.path.join(
        config['init_path'],
        config['single_model']['pkl'],
        f"{model_name}_{pipeline_name}.pkl")
# open model    
with open(model_path, "rb") as file:
    model = pickle.load(file)

In [7]:
results = SelectSingleFeature(model, metric='roc_auc', X_train=X_val, y_train=y_val)

c:\Users\gustavo\anaconda3\envs\ml\Lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\gustavo\anaconda3\envs\ml\Lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\gustavo\anaconda3\envs\ml\Lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\gustavo\anaconda3\envs\ml\Lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please ch

In [8]:
pd.DataFrame(results).sort_values(axis=1, by=0)

,numerical_pipe_dis__FamilySize,categorical_pipe__Embarked,numerical_pipe_con__Age,categorical_pipe__Cabin_1p,numerical_pipe_dis__SibSp,numerical_pipe_dis__IsAlone,categorical_pipe__Pclass,numerical_pipe_con__Fare,categorical_pipe__Sex,categorical_pipe__Title
0,0.521627,0.541532,0.552786,0.603865,0.605238,0.645794,0.704881,0.713444,0.770159,0.775595
1,0.020759,0.062617,0.014492,0.022579,0.069535,0.061857,0.041462,0.072510,0.028533,0.024811


In [7]:
pred_proba = model.predict_proba(X_val)[:,1]